In [ ]:
# ===== standalone: train vs test accuracy + McNemar (no retraining) =====
import itertools, joblib, numpy as np, pandas as pd
from scipy.stats import binomtest
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

DS = "A"   # <- "B" for Dataset B
FEAT, MDIR, RDIR, TAG = {"A": ("DA_features", "models_final_E1", "results_final_E1", "E1"),
                         "B": ("DB_features", "models_final_E2", "results_final_E2", "E2")}[DS]
names = ["decision_tree", "random_forest", "xgboost", "stacking"]

df = pd.read_csv(f"{FEAT}/features.csv")
url_cols  = [c for c in df.columns if c.startswith("url_")]
html_cols = [c for c in df.columns if c.startswith("html_")]
js_cols   = [c for c in df.columns if c.startswith("js_") or c.startswith("tfidf_bpe_")]
cols = url_cols + html_cols + js_cols            # same order as the experiment notebooks
df[cols] = df[cols].fillna(0)

train, test = df[df["split"] == "train"], df[df["split"] == "test"]
y_train, y_test = train["label"].values, test["label"].values
scaler = MinMaxScaler().fit(train[cols])         # scaler wasn't saved; refitting on train is deterministic
X_train, X_test = scaler.transform(train[cols]), scaler.transform(test[cols])

models = {m: joblib.load(f"{MDIR}/{m}_{TAG}_full.joblib") for m in names}
pred_te = {m: models[m].predict(X_test) for m in names}

# 1) train vs test accuracy
acc = pd.DataFrame([{"model": m,
                     "train_acc": accuracy_score(y_train, models[m].predict(X_train)),
                     "test_acc": accuracy_score(y_test, pred_te[m])} for m in names])
acc["gap"] = acc.train_acc - acc.test_acc
acc.round(4).to_csv(f"{RDIR}/train_vs_test_{DS}.csv", index=False)
display(acc.round(4))

# 2) exact McNemar on every pair, Holm-adjusted
rows = []
for a, b in itertools.combinations(names, 2):
    ca, cb = pred_te[a] == y_test, pred_te[b] == y_test
    n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    p = binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if (n01 + n10) else 1.0
    rows.append({"model_a": a, "model_b": b, "only_a_correct": n01, "only_b_correct": n10, "p": p})
mc = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
k = len(mc)
mc["p_holm"] = np.minimum(np.maximum.accumulate([(k - i) * p for i, p in enumerate(mc.p)]), 1.0)
mc["significant_0.05"] = mc.p_holm < 0.05
mc.round(4).to_csv(f"{RDIR}/mcnemar_{DS}.csv", index=False)
display(mc.round(4))

,model,train_acc,test_acc,gap
0,decision_tree,0.9959,0.9939,0.0020
1,random_forest,1.0000,0.9971,0.0029
2,xgboost,0.9996,0.9954,0.0043
3,stacking,0.9999,0.9968,0.0030


,model_a,model_b,only_a_correct,only_b_correct,p,p_holm,significant_0.05
0,decision_tree,stacking,1,11,0.0063,0.0381,True
1,decision_tree,random_forest,2,13,0.0074,0.0381,True
2,xgboost,stacking,0,5,0.0625,0.2500,False
3,random_forest,xgboost,7,1,0.0703,0.2500,False
4,decision_tree,xgboost,2,7,0.1797,0.3594,False
5,random_forest,stacking,2,1,1.0000,1.0000,False
